In [2]:
from training_utilities_2nd_part import *

In [3]:
# weather

from variables_to_specify_weather import *
df, columns_to_normalize, weather_target_col, forecast_avg_target_col_name, avg_target_col_name, No_of_datapoints_in_one_day, start_date, end_date, delta, one_month_days, out_columns, weather_drop_columnss, weather_windows, index_of_one_month, one_month_window_size = variables_to_specify_weather()

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])
df = df.dropna().reset_index(drop=True)
convert_time(df, 'Date Time')
weather_df = df
weather_time_steps = 1


# stationary

In [5]:
weather_len_of_training_data_of_stationary_model =15*No_of_datapoints_in_one_day

train = df[0:weather_len_of_training_data_of_stationary_model] 
test = df[weather_len_of_training_data_of_stationary_model:]

eval_df_first_month, stationary_model1 = stationary_model_with_hptuning(train, test, one_month_window_size, 2, out_columns, weather_target_col, weather_drop_columnss)

sum_training_time_stat1 = eval_df_first_month['training_time'].sum()
print('sum_training_time is: ', sum_training_time_stat1)

print(eval_df_first_month['Testing Error'].mean())
print(eval_df_first_month['mae'].mean())

Model Type: XGBRegressor
Storage Required: 0.08 MB
model storage is : 0.07732009887695312


total_time is:  0.13671129100000012
sum_training_time is:  0.867604170000007
0.01509082037971828
0.07239004683005772


# Model reuse

In [6]:
# Model reuse
daily_df_avg = get_elect_daily_avg(weather_df, No_of_datapoints_in_one_day, weather_target_col, avg_target_col_name)


seasonality_periods_acf_ls, seasonality_periods_acf, segmented_daily_df_avg, filtered_most_similar_dict_wass, filtered_most_similar_dict_tvd, forecast_daily_df_avg, segmented_forecast_daily_df_avg, filtered_forecasted_most_similar_dict_wass, filtered_forecasted_most_similar_dict_tvd = get_seasonality_segments_and_similarities(daily_df_avg, avg_target_col_name, forecast_avg_target_col_name, 15)

Detected seasonality periods (ACF): [15 31 38]
median_value is:  31


# data drift detection

In [7]:
df_copy = weather_df[[weather_target_col]]
target_col = weather_target_col
time_steps = weather_time_steps

df_copy['date'] = pd.to_datetime(df_copy.index)
multiplier = No_of_datapoints_in_one_day
x = 15* multiplier
window_len_=[x]

drift_results_df_ls = []

for i in window_len_:
    start_drift_detection_time = timeit.default_timer()
    drift_results_df = detect_drift_univariate(
        df_copy,
        target_col=weather_target_col,
        window_lengths=window_len_,
        arima_order=(1, 0, 0)
    )
    drift_results_df_ls.append(drift_results_df)
    drift_detection_time = timeit.default_timer() - start_drift_detection_time
    num_true = drift_results_df['drift_detected'].sum()
    print("i is: ", i, " and the Number of True values in 'drift_detected':", num_true, " total number of rows are : ", len(drift_results_df))
    print("drift detection time is: ", drift_detection_time)
    drift_results_df = drift_results_df_ls[0]
    drift_indices = list(drift_results_df.index[drift_results_df['drift_detected']])
    print("indices are: ", drift_indices)

Fold 0: Train size=432, Test size=432
Fold 1: Train size=864, Test size=432
Fold 2: Train size=1296, Test size=432
Fold 3: Train size=1728, Test size=432
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=432, Test size=432
Fold 1: Train size=864, Test size=432
Fold 2: Train size=1296, Test size=432
Fold 3: Train size=1728, Test size=432
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=432, Test size=432
Fold 1: Train size=864, Test size=432
Fold 2: Train size=1296, Test size=432
Fold 3: Train size=1728, Test size=432
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=432, Test size=432
Fold 1: Train size=864, Test size=432
Fold 2: Train size=1296, Test size=432
Fold 3: Train size=1728, Test size=432
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=432, Test size=432
Fold 1: Train size=864, Test size=432
Fold 2: Train size=1296, Test size=432
Fold 3: Train size=1728, Test size=432
Skipping fold 4: I

In [8]:
eval_df_monthly2, avg_ml_storage1 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_wass, stationary_model1, weather_len_of_training_data_of_stationary_model,weather_df, "SA", weather_target_col, weather_drop_columnss, weather_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)


window is:  2160
i/window is :  1.0
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  4320
i/window is :  2.0
similar_month_index is :  0
month_index:  2




window is:  6480
i/window is :  3.0
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  8640
i/window is :  4.0
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  10800
i/window is :  5.0
similar_month_index is :  2
previous_model_i is :  10800
math.floor(previous_model_i/window) is:  5
len(models_ls) is: 4
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  12960
i/window is :  6.0
similar_month_index is :  3
month_index:  6




window is:  15120
i/window is :  7.0
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  17280
i/window is :  8.0
similar_month_index is :  6
previous_model_i is :  17280
math.floor(previous_model_i/window) is:  8
len(models_ls) is: 7
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  19440
i/window is :  9.0
similar_month_i

In [9]:
eval_df_monthly2, avg_ml_storage2 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_tvd, stationary_model1, weather_len_of_training_data_of_stationary_model,weather_df, "SA", weather_target_col, weather_drop_columnss, weather_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  2160
i/window is :  1.0
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  4320
i/window is :  2.0
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  6480
i/window is :  3.0
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  8640
i/window is :  4.0
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  10800
i/window is :  5.0
similar_month_index is :  1
month_index:  5



window is:  12960
i/window is :  6.0
similar_month_index is :  0
month_index:  6




window is:  15120
i/window is :  7.0
similar_month_index is :  0
month_index:  7




window is:  17280
i/window is :  8.0
similar_month_index is :  3
month_index:  8




window is:  19440
i/window is :  9.0
similar_month_index is :  7
previous_model_i is :  19440
math.floor(previous_model_i/window) is:  9
len(models_ls) is: 8
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  21600
i/window is :  10.0
similar_month_index is :  7
previous_model_i is :  21600

In [10]:
eval_df_monthly2, avg_ml_storage3 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_wass, stationary_model1, weather_len_of_training_data_of_stationary_model,weather_df, "ES", weather_target_col, weather_drop_columnss, weather_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  2160
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  4320
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  6480
similar_month_index is :  0
month_index:  2




window is:  8640
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  10800
similar_month_index is :  2
previous_model_i is :  10800
math.floor(previous_model_i/window) is:  5
len(models_ls) is: 4
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  12960
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  15120
similar_month_index is :  0
month_index:  6




window is:  17280
similar_month_index is :  3
month_index:  7




window is:  19440
similar_month_index is :  1
month_index:  8




window is:  21600
similar_month_index is :  5
month_index:  9




window is:  23760
similar_month_index is :  8
previous_model_i is :  23760
math.floor(previous_model_i/window) is:  11
len(models_ls) is: 10
Model Type: XGBRegressor
Storage Required: 0.08 MB


wind

In [11]:
eval_df_monthly2, avg_ml_storage4 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_tvd, stationary_model1, weather_len_of_training_data_of_stationary_model,weather_df, "ES", weather_target_col, weather_drop_columnss, weather_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  2160
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  4320
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  6480
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  8640
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  10800
Model Type: XGBRegressor
Storage Required: 0.07 MB


window is:  12960
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  15120
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  17280
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  19440
similar_month_index is :  2
month_index:  8




window is:  21600
similar_month_index is :  7
month_index:  9




window is:  23760
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  25920
similar_month_index is :  8
previous_model_i is :  25920
math.floor(previous_model_i/window) is:  12
len(models_ls) is: 11
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  28080
Model Type: XGBRegres

In [12]:
avg_ml_storage_reuse = (avg_ml_storage1+avg_ml_storage2+avg_ml_storage3+avg_ml_storage4)/4
print(avg_ml_storage_reuse)

0.08197541236877441


# informed

In [13]:
informed_update(stationary_model1,weather_df, target_col, weather_drop_columnss,time_steps, seasonality_periods_acf,No_of_datapoints_in_one_day, drift_indices, 2)

window is:  2160
Model Type: XGBRegressor
Storage Required: 0.08 MB
window is:  4320
Model Type: XGBRegressor
Storage Required: 0.08 MB
window is:  6480
Model Type: XGBRegressor
Storage Required: 0.08 MB
window is:  8640
Model Type: XGBRegressor
Storage Required: 0.08 MB
window is:  10800
Model Type: XGBRegressor
Storage Required: 0.07 MB
window is:  12960
window is:  15120
Model Type: XGBRegressor
Storage Required: 0.10 MB
window is:  17280
Model Type: XGBRegressor
Storage Required: 0.09 MB
window is:  19440
Model Type: XGBRegressor
Storage Required: 0.09 MB
window is:  21600
window is:  23760
Model Type: XGBRegressor
Storage Required: 0.09 MB
window is:  25920
window is:  28080
window is:  30240
Model Type: XGBRegressor
Storage Required: 0.08 MB
window is:  32400
window is:  34560
Model Type: XGBRegressor
Storage Required: 0.09 MB
window is:  36720
Model Type: XGBRegressor
Storage Required: 0.08 MB
window is:  38880
Model Type: XGBRegressor
Storage Required: 0.10 MB
window is:  41040

# periodical

In [14]:
periodical_retraining_with_hptuning(2, weather_df, weather_windows, out_columns, weather_target_col, weather_drop_columnss)

window is : 720
window size is :  720
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0

([         Training dataset     Testing dataset       mae       mse      rmse  \
  0   trained on window i-1  tested on window i  0.008612  0.000291  0.017053   
  1   trained on window i-1  tested on window i  0.003696  0.000133  0.011543   
  2   trained on window i-1  tested on window i  0.062666  0.005502  0.074175   
  3   trained on window i-1  tested on window i  0.030458  0.001731  0.041606   
  4   trained on window i-1  tested on window i  0.017024  0.000819  0.028614   
  ..                    ...                 ...       ...       ...       ...   
  67  trained on window i-1  tested on window i  0.001312  0.000013  0.003563   
  68  trained on window i-1  tested on window i  0.009515  0.000316  0.017776   
  69  trained on window i-1  tested on window i  0.005852  0.000124  0.011146   
  70  trained on window i-1  tested on window i  0.011143  0.000347  0.018619   
  71  trained on window i-1  tested on window i  0.010872  0.000221  0.014876   
  
            r2       mape